In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import plotly.express as px
from flipside import Flipside
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
import time
from memory_profiler import profile
import json
import csv
import os
import logging
import sys
from importlib import reload
from pymongo import MongoClient
from web3 import Web3
import psycopg2
from psycopg2.extras import execute_values
from io import StringIO
from typing import Any, Dict
from keys import KEYS

In [17]:
# logging configurations
reload(logging)
logging.basicConfig(
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

In [18]:
# Initilize Flipside Client
flipside_key = KEYS['flipside_key']
flipside = Flipside(flipside_key, "https://api-v2.flipsidecrypto.xyz")

In [31]:
def createQueryRun(query : str, api_key:str = flipside_key) -> str :
    
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }

    # Request payload
    payload = {
        "jsonrpc": "2.0",
        "method": "createQueryRun",
        "params": [
            {
                "resultTTLHours": 1,
                "maxAgeMinutes": 0,
                "sql": query ,
                "tags": {
                    "source": "postman-demo",
                    "env": "test"
                },
                "dataSource": "snowflake-default",
                "dataProvider": "flipside"
            }
        ],
        "id": 1
    }

    # Submit createQueryRun request
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        if response.status_code == 200:
            #logging.info("Query run created successfully!")
            logging.debug(response.json())  # Output the response
            return  response.json()['result']['queryRequest']['queryRunId']
        else:
            raise requests.exceptions.HTTPError(
                f"Unexpected status code: {response.status_code}. Details: {response.text}" )
    except Exception as e:
        logging.error(f" createQueryRun Error: {e}")

In [6]:
def getQueryRun(queryRunId:str , api_key:str = flipside_key) -> str:
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }
    
    payload = {
    "jsonrpc": "2.0",
    "method": "getQueryRun",
    "params": [
        {
            "queryRunId": queryRunId
        }
    ],
    "id": 1
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        logging.debug(f'getQueryRun state {response.json()['result']['queryRun']['state']}')
        return response.json()['result']['queryRun']['state']
        
    except Exception as e:
        logging.error(f" getQueryRun Error: {e}")

In [7]:
def queryresult_Pagination(queryRunId:str, page_size:int = 70000) -> list:
       
    current_page_number = 1
    total_pages = 3

    all_rows = []

    while current_page_number <= total_pages:

        try:
            results = flipside.get_query_results(
                queryRunId,
                page_number=current_page_number,
                page_size=page_size         
            )
       
            if results.records:
                total_pages = results.page.totalPages
                all_rows.extend(results.records)
                logging.debug(f"Current page number: {current_page_number} Total Pages: {total_pages}, Rows Retrieved: {len(results.records)}")
            else: 
                logging.warning('No record')
                break

        except Exception as e:
            logging.error(f" Pagination Error: {e}")
            return None
        
        current_page_number += 1

    logging.info(f"Total Pages: {total_pages}, Rows Retrieved: {len(all_rows)}")
        
        
    return all_rows

In [8]:
def extract_flipsidecrypto_data(query:str, api_key = flipside_key, retry_time:int = 90 ,timeout:int = 600 ) -> list:
    
    try:
        query = query
        queryRunId = createQueryRun(query,api_key)

        state = None
        start_time = time.time()

        while state != 'QUERY_STATE_SUCCESS':
            
            state = getQueryRun(queryRunId,api_key)

            if state == 'QUERY_STATE_SUCCESS':
                 break 

            elif state in ['QUERY_STATE_FAILED', 'QUERY_STATE_CANCELED']:
                raise RuntimeError(f"Query execution failed or was canceled. State: {state}")
            
            elif state in ['QUERY_STATE_STREAMING_RESULTS', 'QUERY_STATE_RUNNING', 'QUERY_STATE_READY']:
                if time.time() - start_time > timeout:
                    raise TimeoutError("Query execution exceeded timeout limit.")
                
                logging.info(f"Wainting query excution")
                logging.debug(f"retry after {retry_time} sec")

                time.sleep(retry_time)

            else: raise ValueError(f"Unexpected query state: {state}")

            
        result = queryresult_Pagination(queryRunId)

    except TimeoutError as e:
        logging.error(f"Timeout Error: {e}")
        return None
    except RuntimeError as e:
        logging.error(f"Runtime Error: {e}")
        return None
    except Exception as e:
        logging.error(f" state Error: {e}")
        return None
    
                   

    return result

In [13]:
def fetch_positionsData(pool_address, query) -> list:
    try: 
        logging.info(f"querying data for pool : {pool_address}")
        position_data = extract_flipsidecrypto_data(query)
        logging.info(f"position data fetched successfully for pool {pool_address}, Rows Retrieved: {len(position_data)}")
    except Exception as e:
        logging.error(f"Error fetching position data for pool: {pool_address}: {e}")
    return position_data

In [ ]:
pool_address = '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640'
positin_data_query = f"""
WITH LiquidityPools_Events as (
        SELECT 
        BLOCK_NUMBER ,
        BLOCK_TIMESTAMP,
        TX_HASH,
        EVENT_INDEX,
        CONTRACT_ADDRESS,
        ORIGIN_FROM_ADDRESS,
        ORIGIN_TO_ADDRESS,
        DATA,
        TOPICS[0] as topic0,
        TOPICS,
        CASE TOPICS[0]::STRING 
        WHEN '0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c' THEN 'Burn'
        WHEN '0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde' THEN 'Mint'
        END AS Event_Name,
        '{pool_address}' as pool_address
        FROM ethereum.core.ez_decoded_event_logs log
        WHERE 
        CONTRACT_ADDRESS = '{pool_address}' 
        AND TOPICS[0]::STRING IN (
                    '0x0c396cd989a39f4459b5fa1aed6a9a8dcdbc45908acfd67e028cd568da98982c', -- Burn
                    '0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde' -- Mint
                )
        AND TX_STATUS = 'SUCCESS'
        AND EVENT_REMOVED = 'false'
        AND BLOCK_TIMESTAMP < '2024-10-05'
        ),
        NFTpositions_Events as (
        SELECT 
        BLOCK_NUMBER ,
        BLOCK_TIMESTAMP,
        TX_HASH,
        EVENT_INDEX,
        CONTRACT_ADDRESS,
        ORIGIN_FROM_ADDRESS,
        ORIGIN_TO_ADDRESS,
        DATA,
        TOPICS[0] as topic0,
        TOPICS,
        CASE TOPICS[0]::STRING 
        WHEN '0x3067048beee31b25b2f1681f88dac838c8bba36af25bfb2b7cf7473a5847e35f' THEN 'IncreaseLiquidity'
        WHEN '0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2bff30c67f8dcf9d2377b4' THEN 'DecreaseLiquidity'
        END AS Event_Name,
        '{pool_address}' as pool_address
        FROM ethereum.core.ez_decoded_event_logs log
        WHERE TOPICS[0]::STRING IN (
                    '0x3067048beee31b25b2f1681f88dac838c8bba36af25bfb2b7cf7473a5847e35f', -- IncreaseLiquidity
                    '0x26f6a048ee9138f2c0ce266f322cb99228e8d619ae2bff30c67f8dcf9d2377b4' -- DecreaseLiquidity
                )
        AND TX_STATUS = 'SUCCESS'
        AND EVENT_REMOVED = 'false' 
        AND TX_HASH IN (SELECT TX_HASH FROM LiquidityPools_Events)
        ),
        PositionData As (
        SELECT * FROM LiquidityPools_Events
        UNION ALL 
        SELECT * FROM NFTpositions_Events
        ),
    Final_agg as (
        SELECT 
        log.* ,
        tx.POSITION as TX_INDEX,
        tx.BLOCK_HASH as BLOCK_HASH
        From PositionData log
        LEFT JOIN (
            SELECT BLOCK_HASH,TX_HASH, POSITION
            FROM ethereum.core.fact_transactions
            WHERE TX_HASH IN (SELECT TX_HASH FROM LiquidityPools_Events)
                ) tx  ON (log.TX_HASH = tx.TX_HASH) 
        
                ), 
    Final as (
        SELECT 
        BLOCK_HASH,
        BLOCK_NUMBER ,
        TX_HASH,
        TX_INDEX,
        EVENT_INDEX,
        BLOCK_TIMESTAMP,
        CONTRACT_ADDRESS,
        ORIGIN_FROM_ADDRESS,
        ORIGIN_TO_ADDRESS,
        DATA,
        topic0,
        TOPICS,
        Event_Name,
        pool_address
        FROM Final_agg 
        )
    SELECT
    pool_address,
    BLOCK_HASH,  
    BLOCK_NUMBER,
    TX_HASH,
    TX_INDEX,
    CONTRACT_ADDRESS,
    EVENT_INDEX,
    BLOCK_TIMESTAMP,
    ORIGIN_FROM_ADDRESS,
    ORIGIN_TO_ADDRESS,
    TOPIC0,
    EVENT_NAME,
    TOPICS,
    DATA 
    FROM Final
"""

In [ ]:
positions_extracted_data = fetch_positionsData(pool_address, positin_data_query)

2024-10-05 19:05:22 - INFO - querying data for pool : 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
2024-10-05 19:05:23 - INFO - Query run created successfully!
2024-10-05 19:05:23 - INFO - Wainting query excution
2024-10-05 19:06:54 - INFO - Wainting query excution
2024-10-05 19:08:24 - INFO - Wainting query excution
2024-10-05 19:18:22 - INFO - Total Pages: 7, Rows Retrieved: 453982
2024-10-05 19:18:22 - INFO - position data fetched successfully for pool 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640, Rows Retrieved: 453982


In [ ]:
# positions_extracted_data -> list[dict{}] 
positions_extracted_data[1]

{'pool_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'block_hash': '0x372d2d52fcd3782a07ab6c69e7a2d9bd37b34bf60e6e007aac653d26deee577b',
 'block_number': 12634520,
 'tx_hash': '0xec1f5984a7d86328b40327057e7cfffdc03cac2022caf91856a80367c8acc2fe',
 'tx_index': 102,
 'contract_address': '0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640',
 'event_index': 123,
 'block_timestamp': '2021-06-14T19:39:05.000Z',
 'origin_from_address': '0x4259667850ac23c27cc12c569450ff2b356091fa',
 'origin_to_address': '0xc36442b4a4522e871399cd717abdd847ab11fe88',
 'topic0': '0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde',
 'event_name': 'Mint',
 'topics': ['0x7a53080ba414158be7ec69b987b5fb7d07dee101fe85488f0853ae16239d0bde',
  '0x000000000000000000000000c36442b4a4522e871399cd717abdd847ab11fe88',
  '0x0000000000000000000000000000000000000000000000000000000000030264',
  '0x0000000000000000000000000000000000000000000000000000000000030732'],
 'data': '0x000000000000000000000000c36442b4a452

In [27]:
pool_info_query = f"""
SELECT
    DISTINCT 
    DECODED_LOG:pool as Pool_address,
    DECODED_LOG:token0 as token0,
    DECODED_LOG:token1 as token1,
    DECODED_LOG:fee as fee,
    DECODED_LOG:tickSpacing as tickSpacing
FROM ethereum.core.ez_decoded_event_logs 
WHERE TOPICS[0] = '0x783cca1c0412dd0d695e784568c96da2e9c22ff989357a2e8b1d9b2b4e6b7118'
AND DECODED_LOG:pool = {pool_address} 
"""

In [ ]:
def fetch_infoData(pool_address, query) -> list:
    try: 
        logging.info(f"querying info data for pool : {pool_address}")
        pool_info_data = extract_flipsidecrypto_data(query)
        logging.info(f"info data fetched successfully for pool {pool_address}")
    except Exception as e:
        logging.error(f"Error fetching info_data for pool: {pool_address}: {e}")
    return pool_info_data

In [ ]:
# need API upgrade
pool_info_data = fetch_infoData(pool_address, pool_info_query)

2024-10-05 20:28:39 - INFO - querying info data for pool : 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
2024-10-05 20:28:40 - INFO - Wainting query excution
2024-10-05 20:30:11 - ERROR - Runtime Error: Query execution failed or was canceled. State: QUERY_STATE_FAILED
2024-10-05 20:30:11 - ERROR - Error fetching info_data for pool: 0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640: name 'position_data' is not defined
